# SatQuery AI - Multimodal Remote Sensing Prototype
System check, environment verification, and interactive inference notebook for InternVL2.5-2B.


In [1]:
# Install all required dependencies for SatQuery AI Prototype
%pip install torch torchvision transformers timm accelerate einops pillow numpy peft joblib sentencepiece

import sys
print("Python Version:", sys.version)
print("Python Executable:", sys.executable)



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Python Version: 3.13.3 (v3.13.3:6280bb54784, Apr  8 2025, 10:47:54) [Clang 15.0.0 (clang-1500.3.9.4)]
Python Executable: /Users/afsarazam/Desktop/ SATQUERY/.venv/bin/python


In [2]:
import torch
import torchvision
import transformers
import timm
import numpy as np
import PIL

print("=== SatQuery Environment Check ===")
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Transformers:", transformers.__version__)
print("Timm:", timm.__version__)
print("NumPy:", np.__version__)
print("Pillow:", PIL.__version__)

# Optional ML libraries
try:
    import peft
    print("PEFT:", peft.__version__)
except ImportError:
    print("PEFT: Not installed (optional for LoRA)")

try:
    import bitsandbytes as bnb
    print("bitsandbytes:", bnb.__version__)
except (ImportError, RuntimeError):
    print("bitsandbytes: Not installed or unsupported on current platform (optional for 4-bit CUDA quantization)")

# Hardware Acceleration Check
if torch.cuda.is_available():
    print("Hardware Acceleration: NVIDIA CUDA GPU 🚀")
    print("GPU Device:", torch.cuda.get_device_name(0))
    print("Total VRAM:", f"{torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    print("Hardware Acceleration: Apple Silicon MPS (Metal Performance Shaders) 🚀")
else:
    print("Hardware Acceleration: CPU Mode")


/Users/afsarazam/Desktop/ SATQUERY/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== SatQuery Environment Check ===
PyTorch: 2.14.0
Torchvision: 0.29.0
Transformers: 5.16.1
Timm: 1.0.29
NumPy: 2.5.3
Pillow: 12.3.0
PEFT: 0.20.0
bitsandbytes: Not installed or unsupported on current platform (optional for 4-bit CUDA quantization)
Hardware Acceleration: Apple Silicon MPS (Metal Performance Shaders) 🚀


In [3]:
import torchvision
import einops
import accelerate

print("torchvision:", torchvision.__version__)
print("einops:", einops.__version__)
print("accelerate:", accelerate.__version__)


torchvision: 0.29.0
einops: 0.8.2
accelerate: 1.14.0


In [4]:
import sys
from pathlib import Path

# Add local InternVL paths if present in the workspace
for local_path in [Path("./InternVL"), Path("./InternVL/internvl_chat"), Path("../InternVL")]:
    if local_path.exists() and str(local_path.resolve()) not in sys.path:
        sys.path.insert(0, str(local_path.resolve()))

print("Primary search path:", sys.path[0])


Primary search path: /Library/Frameworks/Python.framework/Versions/3.13/lib/python313.zip


In [5]:
# Verify or detect InternVL package
try:
    import internvl
    print("Local InternVL package found! ✅")
except ImportError:
    print("Local InternVL directory not present; will load model via transformers AutoModel with trust_remote_code=True ✅")


Local InternVL directory not present; will load model via transformers AutoModel with trust_remote_code=True ✅


In [6]:
# Resolve model class: InternVLChatModel or transformers AutoModel fallback
try:
    from internvl.model.internvl_chat import InternVLChatModel
    print("InternVLChatModel imported from internvl package ✅")
except ImportError:
    from transformers import AutoModel
    InternVLChatModel = AutoModel
    print("Using transformers.AutoModel with trust_remote_code=True ✅")


Using transformers.AutoModel with trust_remote_code=True ✅


In [7]:
# Model identifier (supports local directory or Hugging Face Hub)
from pathlib import Path

local_model_dir = Path("models/InternVL2_5-2B")
if local_model_dir.exists():
    model_path = str(local_model_dir.resolve())
    print(f"Using local model directory: {model_path}")
else:
    model_path = "OpenGVLab/InternVL2_5-2B"
    print(f"Using Hugging Face repository: {model_path}")


Using Hugging Face repository: OpenGVLab/InternVL2_5-2B


In [8]:
import torch
from transformers import BitsAndBytesConfig

# Determine compute device and quantization support
has_cuda = torch.cuda.is_available()
has_mps = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()

quant_config = None
if has_cuda:
    try:
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True
        )
        print("Configured 4-bit NF4 quantization for CUDA GPU")
    except Exception as e:
        print(f"BitsAndBytesConfig unavailable: {e}")
elif has_mps:
    print("Configured float16 for Apple Silicon MPS")
else:
    print("Configured float32 for CPU execution")

print(f"Ready to load: {model_path}")


Configured float16 for Apple Silicon MPS
Ready to load: OpenGVLab/InternVL2_5-2B


In [10]:
import gc
import torch
from pathlib import Path

# Clear previous allocations
if "model" in globals():
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

offload_dir = Path("models/offload")
offload_dir.mkdir(parents=True, exist_ok=True)

if torch.cuda.is_available():
    gpu_total_gib = torch.cuda.get_device_properties(0).total_memory // (1024 ** 3)
    gpu_budget_gib = max(2, gpu_total_gib - 2)
    print(f"Loading on CUDA with {gpu_budget_gib} GiB GPU budget...")
    try:
        model = InternVLChatModel.from_pretrained(
            model_path,
            quantization_config=quant_config,
            device_map="auto",
            max_memory={0: f"{gpu_budget_gib}GiB", "cpu": "24GiB"},
            offload_folder=str(offload_dir),
            low_cpu_mem_usage=True,
            trust_remote_code=True,
        ).eval()
        print("Model loaded on CUDA with 4-bit quantization ✅")
    except Exception as e:
        print(f"Quantized load failed: {e}. Falling back to standard float16...")
        model = InternVLChatModel.from_pretrained(
            model_path,
            torch_dtype=torch.float16,
            device_map="auto",
            low_cpu_mem_usage=True,
            trust_remote_code=True,
        ).eval()
        print("Model loaded on CUDA (float16) ✅")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    print("Loading on Apple Silicon MPS (float16)...")
    model = InternVLChatModel.from_pretrained(
        model_path,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    ).to("mps").eval()
    print("Model loaded on Apple Silicon MPS ✅")
else:
    print("Loading on CPU (float32)...")
    model = InternVLChatModel.from_pretrained(
        model_path,
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    ).eval()
    print("Model loaded on CPU ✅")


Loading on Apple Silicon MPS (float16)...


[transformers] InternLM2ForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


Loading weights: 100%|██████████| 517/517 [00:06<00:00, 74.44it/s]


AttributeError: 'InternVLChatModel' object has no attribute 'all_tied_weights_keys'

In [11]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    trust_remote_code=True,
    use_fast=False
)

print("Tokenizer loaded successfully ✅")


Tokenizer loaded successfully ✅


In [15]:
from pathlib import Path

# Select test remote-sensing image from repository
image_path = Path("photo10.jpg")
if not image_path.exists():
    image_path = Path("photo2.jpg")
if not image_path.exists():
    image_path = Path("photo.jpg")

print("Selected test image:", image_path.resolve())


Selected test image: /Users/afsarazam/Desktop/ SATQUERY/photo10.jpg


In [18]:
from PIL import Image
from torchvision import transforms
import torch

image = Image.open(image_path).convert("RGB")

transform = transforms.Compose([
    transforms.Resize((448, 448)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

# Match device and dtype
if torch.cuda.is_available():
    device = "cuda"
    dtype = torch.float16
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
    dtype = torch.float16
else:
    device = "cpu"
    dtype = torch.float32

pixel_values = transform(image).unsqueeze(0).to(
    device=device,
    dtype=dtype
)

print("Image tensor shape:", pixel_values.shape)
print("Device:", pixel_values.device)
print("Dtype:", pixel_values.dtype)


Image tensor shape: torch.Size([1, 3, 448, 448])
Device: mps:0
Dtype: torch.float16


In [20]:
question = "Describe the terrain and identify any structures, vegetation, or water bodies visible in this satellite image."

generation_config = dict(
    num_beams=1,
    max_new_tokens=100,
    do_sample=False,
)

response = model.chat(
    tokenizer,
    pixel_values,
    question,
    generation_config
)

print("Question:", question)
print("Answer:", response)


NameError: name 'model' is not defined

In [27]:
# Helper to get conversation template (supports internvl package or built-in fallback)
try:
    from internvl.conversation import get_conv_template
    print("get_conv_template imported from internvl.conversation ✅")
except ImportError:
    class ConvTemplate:
        def __init__(self, system_message="", roles=("<|im_start|>user\n", "<|im_start|>assistant\n"), sep="<|im_end|>\n"):
            self.system_message = system_message
            self.roles = roles
            self.messages = []
            self.sep = sep

        def append_message(self, role, message):
            self.messages.append((role, message))

        def get_prompt(self):
            prompt = ""
            if self.system_message:
                prompt += f"<|im_start|>system\n{self.system_message}{self.sep}"
            for role, message in self.messages:
                if message:
                    prompt += f"{role}{message}{self.sep}"
                else:
                    prompt += f"{role}"
            return prompt

    def get_conv_template(template_name):
        return ConvTemplate(
            system_message="You are an AI assistant specialized in satellite and aerial remote sensing image analysis.",
            roles=("<|im_start|>user\n", "<|im_start|>assistant\n"),
            sep="<|im_end|>\n"
        )
    print("Using self-contained ConvTemplate fallback ✅")


Using self-contained ConvTemplate fallback ✅


In [30]:
import torch

def get_confidence(model, tokenizer, pixel_values, question, generation_config):
    if "<image>" not in question:
        question = "<image>\n" + question

    IMG_START_TOKEN = "<img>"
    IMG_END_TOKEN = "</img>"
    IMG_CONTEXT_TOKEN = "<IMG_CONTEXT>"
    num_patches = pixel_values.shape[0]

    model.img_context_token_id = tokenizer.convert_tokens_to_ids(IMG_CONTEXT_TOKEN)
    template_name = getattr(model, "template", "internvl2_5")
    template = get_conv_template(template_name)
    if hasattr(model, "system_message") and model.system_message:
        template.system_message = model.system_message

    sep_token = template.sep.strip() if hasattr(template, "sep") and template.sep else "<|im_end|>"
    eos_token_id = tokenizer.convert_tokens_to_ids(sep_token)
    if eos_token_id is None or eos_token_id < 0:
        eos_token_id = tokenizer.eos_token_id

    template.append_message(template.roles[0], question)
    template.append_message(template.roles[1], None)
    query = template.get_prompt()

    num_image_tokens = getattr(model, "num_image_token", 256)
    image_tokens = (
        IMG_START_TOKEN
        + IMG_CONTEXT_TOKEN * num_image_tokens * num_patches
        + IMG_END_TOKEN
    )
    query = query.replace("<image>", image_tokens, 1)

    model_inputs = tokenizer(query, return_tensors="pt")
    
    # Target device resolution
    if hasattr(model, "language_model") and hasattr(model.language_model, "device"):
        device = model.language_model.device
    elif hasattr(model, "device"):
        device = model.device
    else:
        device = pixel_values.device

    input_ids = model_inputs["input_ids"].to(device)
    attention_mask = model_inputs["attention_mask"].to(device)
    pixel_values = pixel_values.to(device)

    config = generation_config.copy()
    config.update({
        "eos_token_id": eos_token_id,
        "output_scores": True,
        "return_dict_in_generate": True,
    })

    with torch.no_grad():
        output = model.generate(
            pixel_values=pixel_values,
            input_ids=input_ids,
            attention_mask=attention_mask,
            **config,
        )

    response = tokenizer.decode(output.sequences[0], skip_special_tokens=True)
    if "assistant" in response.lower():
        response = response.split("assistant")[-1].strip(": \n")
    elif sep_token in response:
        response = response.split(sep_token)[0].strip()

    # Align scores with the final generated tokens
    sequence = output.sequences[0]
    num_scored_tokens = len(output.scores)
    generated_tokens = sequence[-num_scored_tokens:] if num_scored_tokens else sequence

    token_confidences = []
    for step, score in enumerate(output.scores):
        if step >= len(generated_tokens):
            break
        probabilities = torch.softmax(score.float(), dim=-1)
        token_id = generated_tokens[step].to(score.device)
        token_confidences.append(probabilities[0, token_id].item())

    confidence = (
        sum(token_confidences) / len(token_confidences)
        if token_confidences else 0.0
    )

    print("Generated tokens:", len(generated_tokens))
    print("Scored tokens:", len(token_confidences))
    return response, confidence, token_confidences


In [32]:
response, confidence, token_confidences = get_confidence(
    model,
    tokenizer,
    pixel_values,
    question,
    generation_config,
)
print("Question:", question)
print("Answer:", response)
print("Token probabilities:", [round(value, 6) for value in token_confidences[:10]], "..." if len(token_confidences) > 10 else "")
print("Model Confidence:", f"{confidence * 100:.2f}%")


NameError: name 'model' is not defined

In [33]:
# Install joblib for model serialization
%pip install joblib



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [35]:
import pickle
from pathlib import Path

# Save inference configuration bundle for SatQuery FastAPI backend
output_dir = Path("models")
output_dir.mkdir(parents=True, exist_ok=True)
pkl_path = output_dir / "satquery_inference_config.pkl"

num_image_token = int(getattr(model, "num_image_token", 256))

inference_bundle = {
    "model_path": model_path,
    "tokenizer_path": model_path,
    "generation_config": generation_config.copy(),
    "dataset_root": "data",
    "image_size": 448,
    "num_image_token": num_image_token,
    "quantization": {
        "load_in_4bit": quant_config is not None,
        "compute_dtype": "float16" if (torch.cuda.is_available() or (hasattr(torch.backends, "mps") and torch.backends.mps.is_available())) else "float32",
        "quant_type": "nf4",
        "device_map": "auto" if torch.cuda.is_available() else None,
        "offload_folder": "models/offload",
    },
}

with pkl_path.open("wb") as f:
    pickle.dump(inference_bundle, f, protocol=pickle.HIGHEST_PROTOCOL)

print("PKL saved successfully:", pkl_path.resolve())
print("Model path:", inference_bundle["model_path"])
print("Image tokens:", inference_bundle["num_image_token"])
print("Quantization config saved:", inference_bundle["quantization"])


NameError: name 'model' is not defined